<a href="https://colab.research.google.com/github/lelongc/rac/blob/main/voice_qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title ⚙️ BƯỚC 1: CÀI ĐẶT HỆ THỐNG ALL-IN-ONE
import os
from IPython.display import Audio, display, clear_output

print("⏳ Đang cài đặt thư viện lõi (Chỉ mất 1-2 phút)...")
os.system('pip install -U qwen-tts huggingface_hub pydub')
os.system('apt-get install -y ffmpeg sox libsox-fmt-all')
clear_output()

from qwen_tts import Qwen3TTSModel
import torch
import soundfile as sf
import gc

torch.backends.cudnn.benchmark = True
current_model = None
current_model_name = None

# Hàm tải mô hình thông minh (Chống sập RAM Colab)
def load_qwen_model(model_id):
    global current_model, current_model_name

    # Nếu model đang dùng giống model yêu cầu -> Dùng luôn cho lẹ
    if current_model_name == model_id and current_model is not None:
        return current_model

    # Nếu đang dùng model khác -> Xóa model cũ đi để giải phóng RAM
    if current_model is not None:
        print("🧹 Đang dọn dẹp bộ nhớ RAM...")
        del current_model
        gc.collect()
        torch.cuda.empty_cache()

    print(f"📥 Đang tải Model: {model_id.split('/')[-1]}...")
    current_model = Qwen3TTSModel.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="cuda:0",
        attn_implementation="sdpa"
    )
    current_model_name = model_id
    print("✅ Đã tải xong Model!")
    return current_model

print("✅ HỆ THỐNG ALL-IN-ONE ĐÃ SẴN SÀNG!")


********
********
 
✅ HỆ THỐNG ALL-IN-ONE ĐÃ SẴN SÀNG!


In [2]:
# @title 🎨 CHỨC NĂNG 1: VOICE DESIGN (Tạo giọng mới)

# MÔ TẢ GIỌNG BẠN MUỐN TẠO (Bằng tiếng Anh):
mo_ta_giong = "A young female voice, energetic and happy, reading a fairy tale."

# CÂU NÓI ĐỂ AI ĐỌC THỬ:
cau_noi_thu = "Once upon a time, in a magical forest, there lived a tiny fairy."

# ------------------------------------------------------------------
model = load_qwen_model("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign")

print("🎨 Đang thiết kế giọng theo yêu cầu của bạn...")
with torch.inference_mode():
    w, sr = model.generate_voice_design(text=cau_noi_thu, instruct=mo_ta_giong)

ten_file = "1_VoiceDesign_Output.wav"
sf.write(ten_file, w[0], sr)

clear_output()
print("🎉 Đã tạo giọng thành công!")
display(Audio(ten_file))

🎉 Đã tạo giọng thành công!


In [ ]:
# @title 🗣️ CHỨC NĂNG 2: VOICE CLONE (Nhái giọng từ Audio)

# TÊN FILE GIỌNG MẪU BẠN ĐÃ TẢI LÊN COLAB (Cột bên trái 📁):
file_giong_mau = "yo.wav"

# LỜI THOẠI MÀ NGƯỜI TRONG FILE MẪU ĐANG NÓI:
loi_thoai_giong_mau = "Okay. Yeah. I resent you. I love you."

# VĂN BẢN BẠN MUỐN AI ĐỌC:
van_ban_can_doc = "Xin chào, đây là công nghệ nhái giọng siêu việt của Qwen."

# ------------------------------------------------------------------
if not os.path.exists(file_giong_mau):
    print(f"⚠️ LỖI: Chưa thấy file '{file_giong_mau}'. Vui lòng tải lên!")
else:
    model = load_qwen_model("Qwen/Qwen3-TTS-12Hz-1.7B-Base")

    print("⚡️ Đang học giọng...")
    clone_prompt = model.create_voice_clone_prompt(
        ref_audio=file_giong_mau,
        ref_text=loi_thoai_giong_mau,
        x_vector_only_mode=False
    )

    print("🎙️ Đang thu âm...")
    with torch.inference_mode():
        w, sr = model.generate_voice_clone(text=van_ban_can_doc, voice_clone_prompt=clone_prompt)

    ten_file = "2_VoiceClone_Output.wav"
    sf.write(ten_file, w[0], sr)

    clear_output()
    print("🎉 Đã Clone giọng thành công!")
    display(Audio(ten_file))

In [ ]:
# @title 🌟 CHỨC NĂNG 3: CUSTOM VOICE (Dùng các diễn viên lồng tiếng có sẵn)

# CHỌN TÊN DIỄN VIÊN (Ví dụ tiếng Anh có: "Ryan", "Aiden" / Tiếng Trung có: "Vivian", "Serena"):
ten_dien_vien = "Ryan"

# ĐIỀN CẢM XÚC BẠN MUỐN (Ví dụ: "Very angry", "Whispering", "Very sad", "Laughing"):
cam_xuc = "Speaking very seriously and professionally."

# VĂN BẢN BẠN MUỐN HỌ ĐỌC:
van_ban_can_doc = "The global economic situation is currently undergoing massive changes."

# NGÔN NGỮ (Ví dụ: "English", "Chinese". Nếu để "Auto" AI tự đoán):
ngon_ngu = "English"

# ------------------------------------------------------------------
model = load_qwen_model("Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice")

print(f"🎙️ Diễn viên {ten_dien_vien} đang thu âm...")
with torch.inference_mode():
    w, sr = model.generate_custom_voice(
        text=van_ban_can_doc,
        language=ngon_ngu,
        speaker=ten_dien_vien,
        instruct=cam_xuc
    )

ten_file = "3_CustomVoice_Output.wav"
sf.write(ten_file, w[0], sr)

clear_output()
print(f"🎉 Đã thu âm xong bởi {ten_dien_vien}!")
display(Audio(ten_file))